# Fake News Detector — LIAR dataset

In [1]:
import os, zipfile, urllib.request
import pandas as pd

DATA_DIR = 'data'
URL = 'https://www.cs.ucsb.edu/~william/data/liar_dataset.zip'
ZIP_PATH = os.path.join(DATA_DIR, 'liar_dataset.zip')

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ZIP_PATH):
    print('Pobieram LIAR...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    print('Gotowe.')
else:
    print('Plik już pobrany.')

print(os.listdir(DATA_DIR))

Pobieram LIAR...
Gotowe.
['liar_dataset.zip', 'test.tsv', 'README', 'train.tsv', 'valid.tsv']


Wypisuję kolumny z readme:

In [2]:
COLUMNS = [
    'id', 'label', 'statement', 'subject', 'speaker', 'job_title',
    'state', 'party', 'barely_true_counts', 'false_counts',
    'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts',
    'context'
]

train = pd.read_csv(os.path.join(DATA_DIR, 'train.tsv'), sep='\t', header=None, names=COLUMNS)
valid = pd.read_csv(os.path.join(DATA_DIR, 'valid.tsv'), sep='\t', header=None, names=COLUMNS)
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.tsv'),  sep='\t', header=None, names=COLUMNS)

print(f'train: {train.shape}, valid: {valid.shape}, test: {test.shape}')
print('Kolumny:', list(train.columns))
train.head(5)

train: (10240, 14), valid: (1284, 14), test: (1267, 14)
Kolumny: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']


,id,label,statement,subject,speaker,job_title,state,party,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


## Usuwam kolumny, które nie wnoszą sygnału do detekcji fake news

- **`id`** — identyfikator rekordu
- **`speaker`, `job_title`, `state`, `party`** — dane o mówcy (imie i nazwisko, praca, stan, partia pol.)

In [3]:
cols_to_drop = ['id', 'speaker', 'job_title', 'state', 'party']

train = train.drop(columns=cols_to_drop)
valid = valid.drop(columns=cols_to_drop)
test  = test.drop(columns=cols_to_drop)

## Zmienna celu `y` = `label`

Kolumna `label` to nasza zmienna zależna (target). Sprawdzamy, jakie wartości może przyjmować.

In [4]:
labels_train = sorted(train['label'].unique())

print('Unikalne etykiety:', labels_train)

Unikalne etykiety: ['barely-true', 'false', 'half-true', 'mostly-true', 'pants-fire', 'true']


## Zamiana etykiet tylko na true i false, sprawdzenie wielkości zbiorów

- **`false`** ← `pants-fire`, `false`, `barely-true`
- **`true`** ← `half-true`, `mostly-true`, `true`


In [5]:
LABEL_MAP = {
    'pants-fire':  'false',
    'false':       'false',
    'barely-true': 'false',
    'half-true':   'true',
    'mostly-true': 'true',
    'true':        'true',
}

train['label'] = train['label'].map(LABEL_MAP)
valid['label'] = valid['label'].map(LABEL_MAP)
test['label']  = test['label'].map(LABEL_MAP)

print('Rozkład etykiet po binaryzacji:')
print('train:', train['label'].value_counts().to_dict())
print('valid:', valid['label'].value_counts().to_dict())
print('test :', test['label'].value_counts().to_dict())

Rozkład etykiet po binaryzacji:
train: {'true': 5752, 'false': 4488}
valid: {'true': 668, 'false': 616}
test : {'true': 714, 'false': 553}


## Mapowanie etykiet `'true'`/`'false'` → `1`/`0`

In [6]:
LABEL_TO_INT = {'false': 0, 'true': 1}

train['label'] = train['label'].map(LABEL_TO_INT)
valid['label'] = valid['label'].map(LABEL_TO_INT)
test['label']  = test['label'].map(LABEL_TO_INT)

print(train['label'].value_counts().to_dict())

{1: 5752, 0: 4488}


## Preprocessing tekstu — wersja do porównania

Tworzę drugą wersję zbioru danych - z preprocessingiem (lowercase, usunięcie URL-i, interpunkcji, cyfr w słowach). Później chcę trenować RoBERTę na obu wersjach (surowej i przetworzonej) i porównać wyniki. ! Wazne ze interpunkcja wpływa na styl wypowiedzi, więc pytanie czy to nie obnizy jakosci wyniku ???

! todo: zobaczyc ilosc tokenow na wersji preprocessed, moze bedzie mozna wziac mniejsza ilosc niz 128

In [7]:
import re
import string

def preprocess(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_prep = train.copy()
valid_prep = valid.copy()
test_prep  = test.copy()

train_prep['statement'] = train_prep['statement'].apply(preprocess)
valid_prep['statement'] = valid_prep['statement'].apply(preprocess)
test_prep['statement']  = test_prep['statement'].apply(preprocess)

print('PRZED:', train['statement'].iloc[0])
print('PO:   ', train_prep['statement'].iloc[0])

PRZED: Says the Annies List political group supports third-trimester abortions on demand.
PO:    says the annies list political group supports thirdtrimester abortions on demand


In [8]:
%pip install -q transformers torch

wzielam pre trained roberta-base; 2 labelki bo zmienilam y na 2 klasy:

In [10]:
from transformers import RobertaForSequenceClassification

model_roberta = RobertaForSequenceClassification.from_pretrained('roberta-base',num_labels=2)
# 'roberta-base' : 12 layer, 768 hidden, 12 heads, 125M params RoBERTa using BERT-base architecture

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Do tokenizacji uzywam  tokenizatora AutoTokenizer (podobno szybszy niz RobertaTokenizer + mozna szybko zmienic na inny tokenizer)

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sprawdzamy ilości tokenów dla wypowiedzi:

In [12]:
import numpy as np

encoded_train = [tokenizer.encode(s) for s in train['statement']]

lengths = [len(e) for e in encoded_train]
print(f'min={min(lengths)}, max={max(lengths)}')

print('pierwsza wypowiedz:')
print('Tekst :', train['statement'].iloc[0])
print('IDs   :', encoded_train[0])
print('Tokeny:', tokenizer.convert_ids_to_tokens(encoded_train[0]))

Token indices sequence length is longer than the specified maximum sequence length for this model (911 > 512). Running this sequence through the model will result in indexing errors


min=4, max=911
pierwsza wypowiedz:
Tekst : Says the Annies List political group supports third-trimester abortions on demand.
IDs   : [0, 104, 4113, 5, 3921, 918, 9527, 559, 333, 4548, 371, 12, 4328, 38417, 17600, 15, 1077, 4, 2]
Tokeny: ['<s>', 'S', 'ays', 'Ġthe', 'ĠAnn', 'ies', 'ĠList', 'Ġpolitical', 'Ġgroup', 'Ġsupports', 'Ġthird', '-', 'tr', 'imester', 'Ġabortions', 'Ġon', 'Ġdemand', '.', '</s>']


In [13]:
arr = np.array(lengths)
print(f'>128 tokenów: {(arr > 128).sum()} zdań')

>128 tokenów: 4 zdań


Widzimy, ze wiekszosc wypowiedzi ma ponizej 128 tokenow, wiec ustalam, ze na wejsciu bedzie taka wielkosc (roBERTa akceptuje od 1 do 512 tokenow). Dla kazdej wypowiedzi mamy 2 wektory (X) - wektor ztokenizowanych slow i wektor attention_mask - 1 oznacza ze jest slowo, 0 - padding.

In [14]:
example = tokenizer(
    train['statement'].iloc[0],
    padding='max_length',
    max_length=128,
    truncation=True,
    return_tensors='pt'
)

print('Tekst:         ', train['statement'].iloc[0])
print('\ninput_ids:     ', example['input_ids'][0].tolist())
print('\nattention_mask:', example['attention_mask'][0].tolist())

Tekst:          Says the Annies List political group supports third-trimester abortions on demand.

input_ids:      [0, 104, 4113, 5, 3921, 918, 9527, 559, 333, 4548, 371, 12, 4328, 38417, 17600, 15, 1077, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


tokenizacja train, test i validation set:

In [17]:
import torch

def tokenize_set(df):
    return tokenizer(
        list(df['statement']),
        padding='max_length',
        max_length=128,
        truncation=True,
        return_tensors='pt'
    )

tokenize_trainset = tokenize_set(train)
tokenize_validset = tokenize_set(valid)
tokenize_testset  = tokenize_set(test)

print('train:', tokenize_trainset['input_ids'].shape)
print('valid:', tokenize_validset['input_ids'].shape)
print('test :', tokenize_testset['input_ids'].shape)

train: torch.Size([10240, 128])
valid: torch.Size([1284, 128])
test : torch.Size([1267, 128])


zamiana kolumn na torchowe tensory:

In [19]:
y_train = torch.tensor(train['label'].tolist())
y_valid = torch.tensor(valid['label'].tolist())
y_test  = torch.tensor(test['label'].tolist())

print('y_train:', y_train.shape)
print('y_valid:', y_valid.shape)
print('y_test :', y_test.shape)

y_train: torch.Size([10240])
y_valid: torch.Size([1284])
y_test : torch.Size([1267])


opakowanie tokenow i etykiet w obiekt dataloader:

In [21]:
class LiarDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            'input_ids':      self.enc['input_ids'][i],
            'attention_mask': self.enc['attention_mask'][i],
            'labels':         self.labels[i]
        }

data loadery:

In [23]:
train_loader = torch.utils.data.DataLoader(
    LiarDataset(tokenize_trainset, y_train),
    batch_size=16,
    shuffle=True #zeby pmodel nie pamietal kolejnosci
)

valid_loader = torch.utils.data.DataLoader(
    LiarDataset(tokenize_validset, y_valid),
    batch_size=16,
    shuffle=False
)

test_loader = torch.utils.data.DataLoader(
    LiarDataset(tokenize_testset, y_test),
    batch_size=16,
    shuffle=False
)

print('train batches:', len(train_loader))
print('valid batches:', len(valid_loader))
print('test batches: ', len(test_loader))

train batches: 640
valid batches: 81
test batches:  80
